# FEniCSx计算框架 / FEniCSx Computational Framework

---

<div style="float: left; clear: both;" align="left">
<img src="https://fenicsproject.org/assets/img/fenics-logo.png" width="180" alt="fenics-logo.png" align=left hspace="5" vspace="5"/>
<br /><br />
FEniCS项目是一个研究和软件项目，旨在创建用于解决偏微分方程的数学方法和软件。这包括创建直观、高效和灵活的软件。该项目于2003年启动，由来自世界各地一些大学和研究机构的研究人员合作开发。有关FEniCS项目的最新进展和更多信息，请访问<a href="https://fenicsproject.org" title="FEniCS网站">FEniCS网站</a>。
<br /><br />
FEniCS项目的最新版本FEniCSx由几个构建模块组成，即Basix、UFL、FFCx和DOLFINx。DOLFINx是FEniCSx的高性能C++后端，网格、函数空间和函数等结构在这里实现。此外，DOLFINx还包含计算密集型功能，如有限元组装和网格细化算法。它还提供了与线性代数求解器和数据结构的接口，如PETSc。UFL是一种高级形式语言，用于描述具有高级数学语法的变分公式。FFCx是FEniCSx的形式编译器，给定用UFL编写的变量公式，它可以生成高效的C代码。Basix是FEniCSx的有限元后端，负责生成有限元基函数。
<br /><br />
</div>

<div style="float: left; clear: both;" align="left">
<img src="https://fenicsproject.org/assets/img/fenics-logo.png" width="180" alt="fenics-logo.png" align=left hspace="5" vspace="5"/>
<br /><br />
The FEniCS Project is a research and software project aimed at creating mathematical methods and software for solving partial differential equations. This includes creating intuitive, efficient, and flexible software. The project was launched in 2003 and is developed collaboratively by researchers from several universities and research institutes around the world. For the latest progress and more information about the FEniCS Project, please visit the <a href="https://fenicsproject.org" title="FEniCS website">FEniCS website</a>.
<br /><br />
The latest version of the FEniCS Project, FEniCSx, is composed of several building blocks, namely Basix, UFL, FFCx, and DOLFINx. DOLFINx is the high-performance C++ backend of FEniCSx, where structures such as meshes, function spaces, and functions are implemented. In addition, DOLFINx contains compute-intensive features such as finite element assembly and mesh refinement algorithms. It also provides interfaces to linear algebra solvers and data structures such as PETSc. UFL is a high-level form language for describing variational formulations with a high-level mathematical syntax. FFCx is the form compiler of FEniCSx; given a variational formulation written in UFL, it can generate efficient C code. Basix is the finite element backend of FEniCSx, responsible for generating finite element basis functions.
<br /><br />
</div>

---

### 参考资料 / References

[The FEniCSx tutorial](https://jorgensd.github.io/dolfinx-tutorial/fem.html)

<!-- bilingual -->

## 安装 scikit-fem / Installing scikit-fem

---

FEniCSx 对 MPI / PETSc 等底层库依赖复杂，安装包动辄 1.5 GB。为了保持镜像轻量，本项目改用 [`scikit-fem`](https://github.com/kinnala/scikit-fem)：一个纯 Python 的有限元库（~3 MB），在常见的 Poisson / 扩散 / 弹性等问题上能够完整替代 FEniCSx 的教学功能。API 和 FEniCSx 概念基本一一对应——`Basis` ↔ `FunctionSpace`，`asm(laplace, basis)` ↔ `ufl.inner(grad(u), grad(v))*dx`。

FEniCSx has heavy external deps (MPI / PETSc), inflating the image by ~1.5 GB. This project instead uses [`scikit-fem`](https://github.com/kinnala/scikit-fem): a ~3 MB pure-Python finite-element library that covers the Poisson / diffusion / elasticity problems this chapter teaches. Its API maps cleanly to FEniCSx concepts (`Basis` ↔ `FunctionSpace`, `asm(laplace, basis)` ↔ `ufl.inner(grad(u), grad(v))*dx`).

```bash
pip install scikit-fem
```

如果你真的需要 FEniCSx（生产级大规模 MPI 并行），请参考官方 Dolfinx Docker 镜像。If you truly need FEniCSx (production-scale MPI parallel runs), see the official Dolfinx Docker image.

<!-- bilingual -->

## Poisson 方程 / Poisson Equation

---

在单位正方形 $\Omega = (0,1)	imes(0,1)$ 上求解

$$-
abla^2 u(\mathbf{x}) = f(\mathbf{x}) \quad 	ext{in } \Omega,$$

齐次 Dirichlet 边界条件 $u = 0$ on $\partial\Omega$，其中 $f(x,y)=1$（均布载荷）。这是有限元方法最经典的示例。

Solve on the unit square $\Omega = (0,1)	imes(0,1)$

$$-
abla^2 u(\mathbf{x}) = f(\mathbf{x}) \quad 	ext{in } \Omega,$$

with homogeneous Dirichlet boundary condition $u = 0$ on $\partial\Omega$, where $f(x,y) = 1$ (uniform load). This is the canonical first FEM example.

<!-- bilingual -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skfem import MeshTri, Basis, ElementTriP1, condense, asm
from skfem.models.poisson import laplace, unit_load

In [ ]:
# Build a triangular mesh on the unit square, refined 4 times.
mesh = MeshTri().refined(4)
print(f'mesh: {mesh.nvertices} vertices, {mesh.nelements} triangles')

# Linear P1 Lagrange basis on the mesh.
basis = Basis(mesh, ElementTriP1())

In [ ]:
# Assemble the stiffness matrix K and load vector f.
K = asm(laplace, basis)          # corresponds to inner(grad(u), grad(v)) * dx
f = asm(unit_load, basis)        # corresponds to inner(1, v) * dx
print('K:', K.shape, '  f:', f.shape)

In [ ]:
# Locate Dirichlet boundary DOFs (entire boundary of the square).
D = basis.get_dofs()
print('Dirichlet DOFs:', len(D))

In [ ]:
# Condense the system with homogeneous Dirichlet BC and solve.
from skfem import solve
u = basis.zeros()
u[D] = 0.0
u = solve(*condense(K, f, D=D, x=u))
print(f'max u = {u.max():.6f}   mean u = {u.mean():.6f}')

In [ ]:
# Plot the solution on the triangular mesh.
fig, ax = plt.subplots(figsize=(5, 4))
ax.tripcolor(mesh.p[0], mesh.p[1], mesh.t.T, u, shading='gouraud')
ax.triplot(mesh.p[0], mesh.p[1], mesh.t.T, color='k', lw=0.3)
ax.set_aspect('equal'); ax.set_title('Poisson solution u(x,y)')
fig.colorbar(ax.collections[0], ax=ax, shrink=0.8)
plt.show()

In [ ]:
# Verify: the FEM solution to -Δu = 1 on the unit square with u=0 on ∂Ω
# has maximum ≈ 0.0737 at the center. Our refined mesh should match this.
center_idx = np.argmin(np.linalg.norm(mesh.p.T - np.array([0.5, 0.5]), axis=1))
print(f'u at (0.5, 0.5) = {u[center_idx]:.6f}  (analytic ≈ 0.0737)')

In [ ]:
# XDMF export was FEniCSx-specific and has been removed; see matplotlib visualisation above.

In [ ]:
# pyvista 3-D surface view was FEniCSx-specific and has been removed.